# 05. 메타 프롬프트 생성 에이전트 (Prompt Generation Agent)

> **왜 프롬프트 자동 생성이 필요한가요?**
>
> 좋은 프롬프트를 작성하는 것은 어렵고 시간이 많이 들어요. LLM이 LLM을 위한 프롬프트를 자동으로 생성하면 **프롬프트 엔지니어링의 시행착오를 줄일** 수 있어요.

> 🔑 **비유**: 프롬프트 자동 생성은 **작곡 AI**와 같아요. 사람이 "슬픈 분위기의 피아노곡"이라고 요청하면 AI가 구체적인 악보(프롬프트)를 만들어주는 것처럼, 대략적인 요구사항을 구체적인 프롬프트로 변환해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `PromptInstructions` Pydantic 모델로 사용자 요구사항을 구조화하고 수집하는 에이전트를 구현할 수 있어요
2. `bind_tools()`를 활용해 LLM이 도구 호출로 구조화된 데이터를 반환하도록 만들 수 있어요
3. 3단계 워크플로우(info → add_tool_message → prompt)를 StateGraph로 구성할 수 있어요
4. `MemorySaver`와 `thread_id`를 이용한 대화형 요구사항 수집 시스템을 만들 수 있어요

## 사전 지식

- StateGraph, 노드(Node), 엣지(Edge) 기본 개념
- Pydantic BaseModel로 데이터 구조 정의하기
- `bind_tools()`와 Tool Calling 개념
- MemorySaver와 thread_id를 이용한 상태 지속
- 이전 노트북: `04-Agent-Simulation.ipynb` - 가상 사용자, 챗봇 평가

## 메타 프롬프트 생성이란?

**메타 프롬프트(Meta Prompt)**는 "프롬프트를 만들기 위한 프롬프트"를 뜻해요. 즉, LLM이 사용자 요구사항을 수집하고 그것을 바탕으로 고품질 시스템 프롬프트를 자동 생성하는 기법이에요.

왜 이런 시스템이 필요할까요? 일반 사용자는 "좋은 프롬프트"를 작성하는 방법을 잘 모르는 경우가 많아요. 이 에이전트는 사용자로부터 목표와 맥락만 받아서 전문가 수준의 프롬프트를 자동 생성해줘요.

### 핵심 구성 요소

| 구성 요소 | 역할 | 설명 |
|-----------|------|------|
| `PromptInstructions` | 데이터 모델 | 요구사항을 Pydantic으로 구조화 |
| `info` 노드 | 요구사항 수집 | 사용자와 대화하며 필요 정보 수집 |
| `add_tool_message` 노드 | 검증 및 중계 | 도구 호출 결과를 처리하고 검증 |
| `prompt` 노드 | 프롬프트 생성 | META_PROMPT 기반으로 최종 생성 |
| `get_state` 함수 | 조건부 라우팅 | 현재 상태에 따라 다음 노드 결정 |
| `MemorySaver` | 대화 기억 | thread_id로 세션 간 대화 유지 |

> 🔑 **핵심 개념**: `bind_tools([PromptInstructions])`를 사용하면 LLM이 자유 형식 텍스트 대신 **구조화된 Pydantic 객체**를 도구 호출 형태로 반환해요. 이 패턴이 이 에이전트의 핵심이에요!

### 워크플로우 개요

```mermaid
flowchart TD
    A[START] --> B[info 노드<br>요구사항 수집]
    B --> C{get_state<br>상태 판단}
    C -->|도구 호출 발생| D[add_tool_message<br>검증 및 피드백]
    C -->|정보 더 필요| B
    C -->|대화 종료| E[END]
    D --> F[prompt 노드<br>프롬프트 생성]
    F --> E

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class B,D,F process
    class E output
    class C decision
```

> 🎯 **강의 포인트**: 이 그래프의 핵심은 `get_state` 함수예요. 마지막 메시지의 타입을 보고 다음 노드를 결정해요. 메시지에 `tool_calls`가 있으면 `add_tool_message`로, 사용자 메시지면 `info`로, AI 응답이면 `END`로 라우팅해요. 이 패턴은 "도구 호출을 조건으로 하는 분기" 설계에서 자주 쓰여요.

## 환경 설정

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

In [ ]:
# LangSmith 추적 설정 (선택 사항)
import os

# LangSmith를 사용하면 에이전트의 실행 흐름을 웹 UI에서 추적할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Prompt-Generation"

## 1. 요구사항 데이터 모델 정의 (PromptInstructions)

먼저 사용자로부터 수집할 정보의 구조를 Pydantic으로 정의해요. 이 모델은 LLM이 도구 호출을 통해 구조화된 데이터를 반환할 때 사용해요.

### 필드 설계 원칙

| 종류 | 필드 | 역할 |
|------|------|------|
| **필수** | `objective` | 프롬프트의 주요 목표 |
| **필수** | `context` | 사용 맥락 및 도메인 |
| **선택** | `output_format` | JSON, 마크다운 등 출력 형식 |
| **선택** | `tone_style` | 공식적, 친근한, 기술적 등 |
| **선택** | `examples` | 원하는 출력 예시 |
| **선택** | `reasoning_approach` | 단계별, 직접적, 창의적 등 |
| **선택** | `persona` | AI가 맡을 역할/전문성 |
| **선택** | `escalation_conditions` | 불확실 상황에서의 대응 방법 |

> 💡 **실무 팁**: 필수 필드(objective, context)와 선택 필드를 구분하는 게 중요해요. 선택 필드에는 `Optional[str] = None`을 쓰면 LLM이 정보가 없을 때 `null`을 반환하고, 정보가 있을 때만 채워줘요. 사용자를 번잡하게 만들지 않으면서도 필요시 상세한 요구사항을 수집할 수 있어요.

In [ ]:
# 요구사항 수집을 위한 데이터 모델 및 LLM 설정
from typing import List, Optional
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage


# ---------------------------------------------------
# PromptInstructions: 요구사항 수집 데이터 모델
# ---------------------------------------------------
# bind_tools()에 전달되어 LLM이 구조화된 형태로 응답할 때 사용해요
class PromptInstructions(BaseModel):
    """프롬프트 생성을 위한 요구사항 데이터 모델
    
    objective와 context는 필수로, 나머지는 선택적으로 수집해요.
    """

    # 핵심 필드 (필수) - LLM이 반드시 채워야 하는 값
    objective: str      # 프롬프트의 주요 목표
    context: str        # 사용 맥락 및 도메인 정보

    # 출력 관련 (선택)
    output_format: Optional[str] = None       # JSON, 마크다운, 구조화된 텍스트 등
    tone_style: Optional[str] = None          # 공식적, 친근한, 기술적 등
    examples: Optional[List[str]] = None      # 원하는 출력 예시들

    # 행동 지침 (선택)
    reasoning_approach: Optional[str] = None   # 추론 방식 (단계별, 직접적 등)
    persona: Optional[str] = None              # AI가 맡을 역할/전문성
    limitations: Optional[List[str]] = None    # 제약사항 및 금지 사항

    # 안전 및 에스컬레이션 (선택)
    safety_guidelines: Optional[List[str]] = None         # 안전/윤리 지침
    escalation_conditions: Optional[List[str]] = None     # 불확실 상황 대응 방법


# ---------------------------------------------------
# LLM 초기화 및 도구 바인딩
# ---------------------------------------------------
# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# Anthropic: "anthropic:claude-sonnet-4-5"
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# PromptInstructions를 도구로 바인딩하면,
# LLM이 정보 수집이 완료되었을 때 이 구조체 형태로 도구 호출을 발생시켜요
llm_with_tool = llm.bind_tools([PromptInstructions])

# PromptInstructions 모델 필드 목록:
for field_name, field_info in PromptInstructions.model_fields.items():
    required = "필수" if field_info.is_required() else "선택"
    print(f"  [{required}] {field_name}: {field_info.annotation}")

## 2. 정보 수집 노드 (info)

이 노드는 사용자와 대화하며 프롬프트 생성에 필요한 정보를 모아요. LLM에게 역할을 부여하는 시스템 메시지 템플릿이 핵심이에요.

### 정보 수집 전략

- **단계적 수집**: 처음에는 핵심 필드(objective, context)만 요청
- **적응형 질문**: 프롬프트 유형에 따라 관련 선택 필드 추가 질문
- **명확한 중단 조건**: 충분한 정보가 모이면 도구 호출로 종료

> ⚠️ **자주 하는 실수**: 시스템 메시지에서 중단 조건을 명확히 정의하지 않으면 LLM이 언제 도구 호출을 해야 할지 몰라 무한히 질문을 이어가요. "필수 필드 수집 완료 시 도구를 호출하라"고 명시해야 해요.

In [ ]:
# ---------------------------------------------------
# 정보 수집 노드 구현
# ---------------------------------------------------
# 시스템 프롬프트: LLM에게 요구사항 수집 전문가 역할 부여
GATHER_TEMPLATE = """You are an expert prompt engineer helping users create high-quality prompts.
Your role is to gather comprehensive information about their prompt requirements.

<instructions>
Collect the following information systematically:

CORE REQUIREMENTS (Always ask for these first):
- Objective: What is the main goal/purpose of this prompt?
- Context: What domain, use case, or situation will this prompt be used in?

OPTIONAL ENHANCEMENTS (Ask based on user needs):
- Output format: JSON, markdown, structured text, etc.
- Tone and style: professional, friendly, technical, creative, etc.
- Examples: Can you provide examples of ideal outputs?
- Reasoning approach: step-by-step, direct, creative, analytical
- Persona: teacher, analyst, consultant, etc.
- Safety guidelines: ethical considerations or safety requirements
- Escalation conditions: when to ask for clarification

STOPPING CRITERIA:
- Once you have objective + context, you have enough to proceed.
- Call the PromptInstructions tool when you have sufficient information.
- Do NOT keep asking if the user wants to proceed - just call the tool.
</instructions>

[IMPORTANT] Conduct the conversation in Korean, but fill the PromptInstructions tool with English."""


def get_messages_info(messages):
    """시스템 메시지와 기존 대화 메시지를 결합해요"""
    # 시스템 메시지를 앞에 붙여서 LLM에게 역할을 부여해요
    return [SystemMessage(content=GATHER_TEMPLATE)] + messages


def info_chain(state):
    """정보 수집 노드: 사용자와 대화하며 요구사항을 수집해요"""
    # 시스템 메시지 + 대화 기록을 LLM에 전달
    messages = get_messages_info(state["messages"])
    # llm_with_tool: PromptInstructions가 바인딩된 LLM
    response = llm_with_tool.invoke(messages)
    return {"messages": [response]}


# info 노드 함수 정의 완료
# 이 노드는 llm_with_tool을 사용해서
# 정보가 충분히 모이면 PromptInstructions 도구 호출을 발생시켜요.

## 3. 프롬프트 생성 노드 (prompt)

수집된 요구사항을 바탕으로 고품질 프롬프트를 생성하는 노드예요. XML 구조화된 META_PROMPT를 사용해서 체계적인 시스템 프롬프트를 만들어요.

### META_PROMPT 구조

META_PROMPT는 LLM에게 "어떤 프롬프트를 어떻게 만들어야 하는지" 지시하는 메타 수준의 프롬프트예요.

| XML 태그 | 역할 |
|----------|------|
| `<task_definition>` | 프롬프트가 달성해야 할 명확한 목표 정의 |
| `<core_principles>` | 7가지 핵심 원칙 (추론 우선, 구조화 등) |
| `<safety_considerations>` | 안전성 및 윤리 가이드라인 |
| `<prompt_structure>` | 최종 프롬프트의 출력 형식 |
| `<escalation_conditions>` | 불명확한 상황 처리 방법 |

> 🔑 **핵심 개념**: `get_prompt_messages()` 함수는 기존 대화에서 도구 호출 정보(`tool_calls`)를 추출해요. `AIMessage`에서 `tool_calls[0]["args"]`를 가져와 META_PROMPT의 `{reqs}` 자리에 채워 넣어요. 이 방식으로 대화 내용을 프롬프트 생성 컨텍스트로 변환해요.

In [ ]:
# ---------------------------------------------------
# META_PROMPT: 프롬프트 생성을 위한 메타 프롬프트
# ---------------------------------------------------
from langchain.messages import AIMessage, HumanMessage, ToolMessage

META_PROMPT = """You are an expert prompt engineer creating high-quality system prompts.
Generate a comprehensive, well-structured prompt based on the requirements below.

<task_definition>
Create a system prompt that will guide an AI to complete the specified task effectively and safely.
</task_definition>

<core_principles>
1. **Understand the Task**: Analyze objective, context, and constraints thoroughly
2. **Structured Clarity**: Use XML tags, headings, and logical organization
3. **Reasoning First**: Encourage step-by-step reasoning before conclusions
4. **Safety by Design**: Include safety guidelines and escalation paths
5. **Actionable Instructions**: Provide specific, clear directives
6. **Example-Driven**: Include relevant examples when beneficial
7. **Output Specification**: Clearly define expected output format and structure
</core_principles>

<safety_considerations>
- Handle ambiguous or potentially harmful requests with explicit instructions
- Define clear boundaries for AI behavior
- Provide escalation conditions for uncertain situations
</safety_considerations>

<prompt_structure>
Structure the output as:

[Task Instruction - Clear opening statement]

# Steps
1. [Specific actionable steps]

# Output Format
[Precise output specification]

# Notes
- [Edge cases and escalation conditions]
</prompt_structure>

<escalation_conditions>
If any requirement is unclear, proceed with reasonable assumptions and note uncertainties.
</escalation_conditions>

# Instructions
Based on these requirements, create an optimized system prompt:

Requirements:
{reqs}"""


def get_prompt_messages(messages: list):
    """대화 기록에서 도구 호출 정보를 추출해 프롬프트 생성 메시지를 구성해요"""
    tool_call = None      # PromptInstructions 도구 호출 args를 담을 변수
    other_msgs = []       # 도구 호출 이후의 추가 메시지들

    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            # AIMessage에 tool_calls가 있으면 첫 번째 호출의 args를 저장
            tool_call = m.tool_calls[0]["args"]
        elif isinstance(m, ToolMessage):
            # ToolMessage는 건너뜀 (도구 실행 결과는 불필요)
            continue
        elif tool_call is not None:
            # 도구 호출 이후에 온 메시지는 추가 컨텍스트로 포함
            other_msgs.append(m)

    # META_PROMPT에 수집된 요구사항을 삽입해서 시스템 메시지 생성
    return [SystemMessage(content=META_PROMPT.format(reqs=tool_call))] + other_msgs


def prompt_gen_chain(state):
    """프롬프트 생성 노드: 수집된 요구사항으로 최종 프롬프트를 만들어요"""
    messages = get_prompt_messages(state["messages"])
    # 일반 llm(도구 없음)으로 최종 프롬프트 생성
    response = llm.invoke(messages)
    return {"messages": [response]}


# prompt 노드 함수 정의 완료
# 이 노드는 도구 호출 args를 META_PROMPT에 삽입해서 최종 프롬프트를 생성해요.

## 4. 조건부 라우팅 함수 (get_state)

이 함수는 현재 대화 상태를 보고 다음에 실행할 노드를 결정해요. 메시지의 타입을 검사해서 3가지 경로 중 하나로 라우팅해요.

### 라우팅 로직

```
마지막 메시지 타입?
├── AIMessage + tool_calls 있음 → "add_tool_message"
├── HumanMessage가 아님 (= AI 응답) → END
└── HumanMessage
    ├── 이전에 tool_calls 있었음 → "prompt"
    └── 그 외 → "info"
```

> 🎯 **강의 포인트**: `get_state` 함수를 라이브로 추적해보면, 대화가 진행될수록 라우팅이 어떻게 변하는지 직접 볼 수 있어요. 처음에는 항상 `info`로 가다가, 요구사항이 충분히 모이면 LLM이 도구 호출을 발생시키고 `add_tool_message` → `prompt` 경로로 전환돼요.

In [ ]:
# ---------------------------------------------------
# 조건부 라우팅 함수: 다음 노드 결정
# ---------------------------------------------------
from langgraph.graph import END


def get_state(state) -> str:
    """현재 상태를 분석해서 다음 실행할 노드를 반환해요
    
    Returns:
        "add_tool_message": 도구 호출 발생 → 검증 단계
        "info": 정보 더 필요 → 계속 수집
        "prompt": 이전에 도구 호출 있었음 → 프롬프트 생성
        END: AI 응답 완료 → 대화 종료
    """
    messages = state["messages"]

    # 메시지가 없는 경우 → 정보 수집 시작
    if not messages:
        return "info"

    last_message = messages[-1]

    # Case 1: 마지막 메시지가 AI 메시지이고 도구 호출 있음
    # → LLM이 요구사항 수집 완료로 판단, 도구 호출 발생
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "add_tool_message"

    # Case 2: 마지막 메시지가 HumanMessage가 아님 (AI 응답)
    # → 사용자 차례, 더 이상 자동으로 진행하지 않음
    elif not isinstance(last_message, HumanMessage):
        return END

    # Case 3: 마지막이 HumanMessage이고, 이전에 도구 호출이 있었음
    # → 사용자가 추가 입력 후 프롬프트 생성 재개
    for msg in reversed(messages[:-1]):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            return "prompt"    # 이전 도구 호출 발견 → 프롬프트 생성
        elif isinstance(msg, ToolMessage):
            continue           # ToolMessage는 건너뜀

    # Case 4: HumanMessage이지만 이전 도구 호출 없음 → 계속 정보 수집
    return "info"


def validate_tool_call(tool_call_args: dict) -> tuple[bool, str]:
    """도구 호출 인수의 유효성을 검사해요
    
    Returns:
        (True, "") - 유효함
        (False, 오류메시지) - 유효하지 않음
    """
    # 필수 필드(objective, context) 존재 여부 확인
    required_fields = ["objective", "context"]
    for field in required_fields:
        if not tool_call_args.get(field):  # None 또는 빈 문자열 체크
            return False, f"필수 필드 '{field}'가 누락되었어요."
    return True, ""


# get_state 라우팅 함수 정의 완료
# validate_tool_call 검증 함수 정의 완료

## 5. StateGraph 그래프 구성

앞서 정의한 노드와 라우팅 함수를 연결해서 전체 워크플로우 그래프를 만들어요.

> 💡 **실무 팁**: `MemorySaver`와 `thread_id`를 조합하면 여러 대화 세션을 독립적으로 관리할 수 있어요. 같은 `thread_id`를 사용하면 이전 대화를 이어받고, 새 `uuid4()`를 생성하면 새로운 대화를 시작해요.

In [ ]:
# ---------------------------------------------------
# StateGraph 구성: 노드와 엣지 연결
# ---------------------------------------------------
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict


# 그래프 상태(State) 정의
class State(TypedDict):
    # add_messages: 새 메시지가 기존 목록에 누적되도록 하는 리듀서
    messages: Annotated[list, add_messages]


# MemorySaver: 대화 기록을 메모리에 저장해서 멀티턴 대화를 지원해요
memory = MemorySaver()

# StateGraph 초기화
workflow = StateGraph(State)

# ---------------------------------------------------
# 노드 추가
# ---------------------------------------------------
workflow.add_node("info", info_chain)          # 요구사항 수집 노드
workflow.add_node("prompt", prompt_gen_chain)  # 프롬프트 생성 노드


@workflow.add_node
def add_tool_message(state: State):
    """도구 메시지 추가 노드: 도구 호출 결과를 검증하고 피드백을 제공해요"""
    try:
        last_message = state["messages"][-1]
        tool_call = last_message.tool_calls[0]  # 첫 번째 도구 호출 가져오기
        tool_args = tool_call["args"]           # 수집된 요구사항 args

        # 필수 필드 유효성 검사
        is_valid, error_msg = validate_tool_call(tool_args)

        if not is_valid:
            # 유효하지 않으면 사용자에게 재입력 요청
            return {
                "messages": [
                    ToolMessage(
                        content=f"요구사항 정보가 불완전해요: {error_msg}. 다시 알려주세요.",
                        tool_call_id=tool_call["id"]
                    )
                ]
            }

        # 유효하면 수집 완료 메시지 반환
        collected_info = f"""요구사항 수집 완료!

수집된 정보:
- 목표(Objective): {tool_args.get('objective', '미지정')}
- 맥락(Context): {tool_args.get('context', '미지정')}
- 출력 형식: {tool_args.get('output_format', '미지정')}
- 톤/스타일: {tool_args.get('tone_style', '미지정')}

이제 최적화된 프롬프트를 생성할게요..."""

        return {
            "messages": [
                ToolMessage(
                    content=collected_info,
                    tool_call_id=tool_call["id"]  # 호출 ID와 매칭 필수
                )
            ]
        }

    except Exception as e:
        # 예외 발생 시 안전하게 처리
        return {
            "messages": [
                ToolMessage(
                    content=f"요구사항 처리 중 오류 발생: {str(e)}",
                    tool_call_id=state["messages"][-1].tool_calls[0]["id"]
                )
            ]
        }


# ---------------------------------------------------
# 엣지 연결
# ---------------------------------------------------
# 시작점 → info 노드 (항상 정보 수집부터 시작)
workflow.add_edge(START, "info")

# info 노드 → 조건부 라우팅
workflow.add_conditional_edges(
    "info",
    get_state,  # 라우팅 함수
    {
        "add_tool_message": "add_tool_message",  # 도구 호출 → 검증
        "info": "info",                          # 정보 더 필요 → 반복
        END: END,                                # AI 응답 후 → 종료
    }
)

# add_tool_message → prompt (항상 프롬프트 생성으로 이동)
workflow.add_edge("add_tool_message", "prompt")

# prompt → END (프롬프트 생성 완료 후 종료)
workflow.add_edge("prompt", END)

# 그래프 컴파일
graph = workflow.compile(checkpointer=memory)

# 그래프 컴파일 완료!

### 그래프 시각화

컴파일된 그래프의 구조를 시각적으로 확인해볼게요.

In [ ]:
# 그래프 흐름: START → info → (get_state 분기) → add_tool_message → prompt → END
# info: 사용자와 대화하며 프롬프트 생성에 필요한 요구사항을 수집해요
# get_state: 마지막 메시지 타입에 따라 add_tool_message / info / END 중 하나로 라우팅해요
# add_tool_message: PromptInstructions 도구 호출 결과를 검증하고 피드백을 제공해요
# prompt: 수집된 요구사항을 META_PROMPT에 삽입하여 최종 프롬프트를 생성해요
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 6. 에이전트 실행 - 단계별 스트리밍

이제 구성한 그래프를 실제로 실행해 볼게요. 먼저 단일 메시지를 보내 에이전트가 어떻게 반응하는지 확인해요.

> 💡 **실무 팁**: 스트리밍 실행 시 `stream_mode="updates"`를 사용하면 각 노드의 출력을 실시간으로 받을 수 있어요. 긴 프롬프트 생성에 유용해요.

In [ ]:
# ---------------------------------------------------
# 단일 메시지 실행 테스트
# ---------------------------------------------------
import uuid

# 새로운 대화 세션을 위한 고유 thread_id 생성
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"새 대화 세션 시작: thread_id = {thread_id[:8]}...\n")
# ============================================================

# 첫 번째 사용자 메시지
user_input = "고객 지원 챗봇을 위한 프롬프트를 만들고 싶어요"
print(f"사용자: {user_input}\n")

# 그래프 실행 - 스트리밍으로 각 노드 출력 확인
for chunk in graph.stream(
    {"messages": [HumanMessage(content=user_input)]},
    config,
    stream_mode="updates"  # 각 노드 실행 결과를 순서대로 받아요
):
    for node_name, node_output in chunk.items():
        print(f"--- 노드: {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                # 메시지 타입에 따라 출력
                if hasattr(msg, 'content') and msg.content:
                    print(msg.content)
        print()

### 대화 이어가기 (멀티턴)

같은 `thread_id`로 계속 대화하면 이전 맥락을 기억해요. 추가 정보를 제공해서 에이전트가 프롬프트를 생성하도록 유도해볼게요.

In [ ]:
# ---------------------------------------------------
# 멀티턴 대화: 추가 정보 제공
# ---------------------------------------------------
# 같은 thread_id로 계속 대화 - 이전 맥락을 기억해요
# ============================================================
user_input2 = "이커머스 사이트 고객 지원용이에요. 한국어로 친근하게 응답하고, 주문/배송/환불 질문을 주로 처리해야 해요"
print(f"사용자: {user_input2}\n")

for chunk in graph.stream(
    {"messages": [HumanMessage(content=user_input2)]},
    config,    # 같은 thread_id → 이전 대화 기억
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        print(f"--- 노드: {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                if hasattr(msg, 'content') and msg.content:
                    print(msg.content)
        print()

## 7. 대화형 프롬프트 생성기

실제 사용 환경처럼 사용자 입력을 받아서 대화하는 함수를 만들어볼게요.

> ⚠️ **자주 하는 실수**: Jupyter 환경에서 `input()` 함수를 쓸 때 무한 루프가 걸릴 수 있어요. 아래 코드는 Jupyter에서 안전하게 실행되도록 고정된 시나리오로 테스트해요. 실제 대화형 CLI를 원하면 `.py` 파일로 변환해서 터미널에서 실행하세요.

In [ ]:
# ---------------------------------------------------
# 대화형 프롬프트 생성기 함수 정의
# ---------------------------------------------------

def run_prompt_generator_demo(scenario: list[str]):
    """시나리오 기반 프롬프트 생성기 데모
    
    Args:
        scenario: 순서대로 입력할 사용자 메시지 목록
    """
    # 새 세션 시작
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    # [프롬프트 생성기 데모]
    # ============================================================

    for i, user_msg in enumerate(scenario, 1):
        print(f"\n[턴 {i}] 사용자: {user_msg}")
        # ----------------------------------------

        for chunk in graph.stream(
            {"messages": [HumanMessage(content=user_msg)]},
            config,
            stream_mode="updates"
        ):
            for node_name, node_output in chunk.items():
                # 노드 이름 표시
                node_labels = {
                    "info": "[정보 수집]",
                    "add_tool_message": "[요구사항 검증]",
                    "prompt": "[프롬프트 생성]"
                }
                label = node_labels.get(node_name, f"[{node_name}]")
                print(f"\n{label}")

                if "messages" in node_output:
                    for msg in node_output["messages"]:
                        if hasattr(msg, 'content') and msg.content:
                            # 긴 내용은 첫 500자만 출력
                            content = msg.content
                            if len(content) > 500:
                                print(content[:500] + "\n...")
                            else:
                                print(content)

        print()


# 테스트 시나리오 1: 블로그 작성 프롬프트
scenario_blog = [
    "블로그 글 작성을 도와주는 프롬프트를 만들고 싶어요",
    "IT 기술 블로그예요. SEO 최적화된 마크다운 형식으로, 초보자도 이해할 수 있게 쉽게 써야 해요",
]

# 시나리오: IT 블로그 작성 프롬프트 생성
run_prompt_generator_demo(scenario_blog)

In [ ]:
# ============================================================
# TODO: 나만의 프롬프트 생성 시나리오를 만들어보세요
# 힌트: scenario_custom 리스트에 원하는 요구사항을 입력하고
#       run_prompt_generator_demo()를 호출하세요
# 예상 결과: 입력한 요구사항에 맞는 커스텀 시스템 프롬프트가 생성돼요
# ============================================================

# 예시 1: 코드 리뷰 어시스턴트
# scenario_custom = [
#     "코드 리뷰를 도와주는 AI 어시스턴트 프롬프트를 만들고 싶어요",
#     "Python 코드 위주예요. 가독성, 성능, 보안 관점에서 리뷰하고 개선안을 제시해야 해요",
# ]

# 예시 2: 데이터 분석 리포트 작성기
# scenario_custom = [
#     "데이터 분석 결과를 비즈니스 리포트로 변환하는 프롬프트를 만들어요",
#     "마케팅팀용이에요. 기술적 용어 최소화, 핵심 인사이트 중심, 차트 설명 포함해야 해요",
# ]

# 직접 시나리오 작성
scenario_custom = [
    # 여기에 첫 번째 메시지를 입력하세요
    "교육용 퀴즈 생성기 프롬프트를 만들고 싶어요",
    # 여기에 추가 요구사항을 입력하세요
    "초등학생 수학 교육용이에요. 객관식 4지선다 형식, 오답에 대한 힌트도 제공해야 해요",
]

# 커스텀 시나리오 실행:
run_prompt_generator_demo(scenario_custom)

## 8. 현재 상태 조회 및 대화 기록 확인

`MemorySaver`를 사용했기 때문에 `graph.get_state()`로 현재 저장된 상태를 조회할 수 있어요.

> 🔑 **핵심 개념**: `get_state(config)`는 특정 `thread_id`의 현재 체크포인트를 반환해요. 여기서 전체 대화 기록을 볼 수 있어요. 이 기능을 활용해서 디버깅이나 대화 이어받기 기능을 구현할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 저장된 대화 상태 조회
# ---------------------------------------------------
# config는 앞서 single-run에서 사용한 thread_id
saved_state = graph.get_state(config)

# 현재 저장된 대화 기록:
print(f"총 메시지 수: {len(saved_state.values['messages'])}\n")

for i, msg in enumerate(saved_state.values["messages"]):
    # 메시지 타입별로 다르게 출력
    msg_type = type(msg).__name__
    content_preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
    print(f"[{i+1}] {msg_type}: {content_preview}")
    
    # AIMessage에 tool_calls가 있으면 표시
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"     [도구 호출] PromptInstructions args 수집됨")

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **PromptInstructions (Pydantic)**: 필수 필드(objective, context)와 선택 필드로 구성된 요구사항 데이터 모델로, `Optional[str] = None` 패턴으로 유연하게 설계했어요
- **bind_tools() 패턴**: `llm.bind_tools([PromptInstructions])`를 사용하면 LLM이 충분한 정보를 모았을 때 자유 텍스트 대신 구조화된 도구 호출을 발생시켜요
- **3단계 워크플로우**: `info(수집)` → `add_tool_message(검증)` → `prompt(생성)` 순서로 진행되는 상태 기계 구조예요
- **get_state 라우팅**: 메시지 타입(AIMessage.tool_calls 유무, HumanMessage 여부)을 검사해서 다음 노드를 결정해요
- **META_PROMPT**: XML 구조화된 메타 프롬프트로 7가지 핵심 원칙, 안전 고려사항, 에스컬레이션 조건을 포함해요
- **MemorySaver + thread_id**: `uuid4()`로 생성한 고유 thread_id를 config에 전달해 세션별 대화 기록을 독립적으로 관리해요

## 다음 노트북 예고

다음 `06-GraphRAG-Neo4j.ipynb`에서는 **Text2Cypher와 Neo4j 그래프 데이터베이스 연동**을 배워요. 자연어 질문을 Cypher 쿼리로 변환하고, Neo4j의 그래프 구조에서 관계형 데이터를 추출하는 GraphRAG 시스템을 구현해볼 거예요. LangGraph로 쿼리 생성 → 실행 → 검증의 파이프라인을 구성해요.

<!-- AUTOPILOT_CREATE_AGENT_DEEP_AGENT_APPENDIX -->
## 보강: `create_agent` + structured output으로 프롬프트 생성기를 단순화하기

### 참고 공식 문서
- [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)

본문은 `bind_tools([PromptInstructions])`와 `StateGraph` 라우팅으로 “정보 수집 → 요구사항 검증 → 최종 프롬프트 생성”을 직접 구현합니다. 이 방식은 tool call 기반 요구사항 수집 원리를 배우기에 좋습니다.

공식 structured output 문서 기준으로는 `create_agent(..., response_format=스키마)`를 사용해 더 간결한 구현을 만들 수 있어요. 이때 LangChain은 모델/프로바이더 능력에 맞춰 provider-native structured output 또는 tool-calling 전략을 선택하고, 검증된 결과를 `result["structured_response"]`에 넣어줍니다.

고도화 방향은 다음과 같습니다.

- 요구사항 수집 결과를 Pydantic 모델로 강제해 후속 UI/저장/평가에 바로 사용
- generated prompt뿐 아니라 test cases, refusal policy, evaluation rubric까지 구조화
- LangSmith 평가에서 `structured_response` 필드를 기준으로 자동 채점
- 단순한 대화 상태 그래프가 필요 없는 경우 `create_agent` 하나로 MVP 구성

아래 코드는 본문 구현의 “간결한 제품형 baseline”입니다. 수업에서는 두 구현을 비교해 **직접 StateGraph를 짜는 경우와 agent abstraction을 쓰는 경우의 trade-off**를 토론할 수 있어요.


In [ ]:
# ============================================================
# 선택 실행: create_agent + structured_response 프롬프트 생성기
# ============================================================
RUN_CREATE_AGENT_PROMPT_APPENDIX = False

if RUN_CREATE_AGENT_PROMPT_APPENDIX:
    from typing import List, Optional

    from langchain.agents import create_agent
    from langchain.chat_models import init_chat_model
    from pydantic import BaseModel, Field

    # --------------------------------------------------------
    # 1) 최종 산출물을 구조화하는 스키마
    # --------------------------------------------------------
    class PromptBlueprint(BaseModel):
        """프롬프트 생성기가 반드시 반환해야 하는 구조화 결과."""
        objective: str = Field(description="프롬프트가 달성해야 하는 핵심 목표")
        target_user: str = Field(description="이 프롬프트를 사용할 사용자/팀")
        system_prompt: str = Field(description="바로 사용할 수 있는 최종 시스템 프롬프트")
        required_context: List[str] = Field(description="실행 전 사용자에게 추가로 받아야 할 정보")
        evaluation_rubric: List[str] = Field(description="생성된 프롬프트 품질을 평가할 기준")
        safety_notes: Optional[str] = Field(default=None, description="안전/보안/거절 정책 관련 메모")

    # --------------------------------------------------------
    # 2) create_agent에 response_format을 직접 전달
    # --------------------------------------------------------
    prompt_architect = create_agent(
        model=init_chat_model("openai:gpt-4o-mini", temperature=0),
        tools=[],
        response_format=PromptBlueprint,
        system_prompt="""
너는 시니어 프롬프트 아키텍트다.
사용자의 모호한 요구를 제품에서 바로 쓸 수 있는 시스템 프롬프트로 변환한다.
반드시 objective, target_user, system_prompt, required_context,
evaluation_rubric, safety_notes를 구조화해서 반환한다.
한국어로 작성하되, 시스템 프롬프트 내부의 역할/규칙은 명확하고 테스트 가능해야 한다.
""",
    )

    # --------------------------------------------------------
    # 3) 실행 예시
    # --------------------------------------------------------
    result = prompt_architect.invoke({
        "messages": [{
            "role": "user",
            "content": "초등학생 수학 퀴즈를 만드는 AI 튜터용 프롬프트를 만들어줘.",
        }]
    })

    blueprint: PromptBlueprint = result["structured_response"]
    import json
    print(json.dumps(blueprint.model_dump(), indent=2, ensure_ascii=False))
